# SoundReg — autoregressive acoustic detection on Acoustic-BEV

Trains **SoundReg** from scratch on the Acoustic-BEV `nn_data` pickles, runs
inference on the four benchmark test splits, and scores with the benchmark
matcher — directly comparable to the rows in `acoustic-bev-benchmark`.

**Method.** SoundReg casts the sounding-object set as a token sequence
(working brief, Eqs. 4–6): each object is two tokens `(t_r, t_theta)` on a
polar grid, a scene is `[BOS, obj_1, ..., obj_N, EOS]` emitted in **dominance
order** — loudest first by steered-response power, so the causal decoder
conditions each (possibly masked) source on the maskers already emitted.
`[EOS]` carries the cardinality: no slot count K, no confidence threshold tau.

**Reuse.** The encoder is the MoG-slot network's front end, re-implemented
faithfully: SpeedFiLM velocity conditioning + the conv trunk that ends in a
`(B, 100, T=19, n_theta=360)` feature map. Where the K-slot head pooled time
and cross-attended K learned queries over the 360 angular tokens, SoundReg
keeps the same angular token sequence and cross-attends a **causal Transformer
decoder** over it instead. Same data, same geometry (mic XML), same front end
— only the head changes.

**Structure.** Settings → Environment → Geometry & dominance → Dataset →
Model → Training → Curves → Inference & export → Evaluation → Result.

## Settings

In [ ]:
from pathlib import Path
import os

# Resolve the repo root robustly, regardless of the notebook's working dir.
REPO = Path.cwd().resolve()
while not (REPO / "notebooks").is_dir() and REPO != REPO.parent:
    REPO = REPO.parent
assert (REPO / "notebooks").is_dir(), f"could not locate repo root from {Path.cwd()}"

# Data: the same nn_data pickles the acoustic-bev-benchmark uses.
# Override with SOUNDREG_DATA when the data lives elsewhere.
DATA_DIR = Path(os.environ.get("SOUNDREG_DATA", "/home/joadeola/Datasets/nn_data"))
TRAIN_PKL = DATA_DIR / "train.pkl"
VAL_PKL   = DATA_DIR / "val.pkl"
SPLITS = {"static_easy":        DATA_DIR / "test_static_easy.pkl",
          "static_difficult":   DATA_DIR / "test_static_difficult.pkl",
          "easy_together":      DATA_DIR / "test_easy_together.pkl",
          "difficult_together": DATA_DIR / "test_difficult_together.pkl"}
MIC_XML = REPO / "metadata" / "mic_on_the_car.xml"   # 32 mics, two-height arrays

# Outputs: run artifacts + benchmark-format prediction/metrics CSVs.
RUN_NAME    = "soundreg_v1"
RUN_DIR     = REPO / "notebooks" / "runs" / RUN_NAME
RESULTS_DIR = REPO / "notebooks" / "results"
METRICS_CSV = RESULTS_DIR / "metrics_soundreg.csv"
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME, MODEL_LABEL, CAPABILITY = "soundreg", "SoundReg (ours)", "both"

# STFT geometry of the pickles (metadata/stft_config.yaml of the benchmark):
# 48 kHz, n_fft 512, hop 256; 84 bins span [100, 8000] Hz; 19 time frames.
SR, N_FFT = 48000, 512
FREQ_MIN_HZ, FREQ_MAX_HZ = 100.0, 8000.0
FIXED_T_FRAMES, FREQ_BINS, N_MICS = 19, 84, 32
MAX_RANGE_M = 30.0
C_SOUND = 343.0

# Polar grid = the target vocabulary. theta in [0, 360) like the benchmark;
# azimuth finer than range: 5 deg bins (= the strictest benchmark gate), 1.25 m bins.
N_R, N_THETA = 24, 72

# Ordering — THE ablation knob: "dominance" | "random" | "near_to_far".
ORDERING = "dominance"

# Model. Trunk numbers are the MoG-slot defaults (do not retune casually:
# the point is reusing the front end as-is). Decoder is the new head.
N_THETA_FEAT = 360                  # angular feature tokens out of the trunk
FILM_HIDDEN = 64
D_MODEL, N_LAYERS, N_HEADS, D_FF, DROPOUT = 192, 4, 6, 768, 0.1
MAX_OBJECTS = 6                     # capacity cap (benchmark scenes have <= 3)

# Training.
EPOCHS, BATCH_SIZE, LR, WEIGHT_DECAY, GRAD_CLIP = 60, 32, 3e-4, 0.01, 1.0
SEED = 42069                        # benchmark-wide fixed seed
DEVICE = "auto"

# Evaluation (the benchmark gates).
ANGLE_GATE, RANGE_GATE = 5.0, 5.0

# Smoke knobs: small ints for a quick end-to-end check, None for a full run.
LIMIT_TRAIN = None
LIMIT_VAL = None
LIMIT_TEST = None
print(f"repo={REPO}\ndata={DATA_DIR}\nrun={RUN_DIR}")

## Environment

In [ ]:
import csv, math, pickle, time
import xml.etree.ElementTree as ET

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
import matplotlib.pyplot as plt

torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE_OBJ = torch.device("cuda:0" if (DEVICE == "auto" and torch.cuda.is_available()) else
                          "cpu" if DEVICE == "auto" else DEVICE)
print("torch", torch.__version__, "| device:", DEVICE_OBJ)

## Geometry and dominance

Mic positions come from the array XML (32 `<pos>` elements). The file's axes
are `(x_fwd, y_up, z_lat)`; annotations live in the ego ground plane with
`theta = atan2(y, x) % 360`, so the horizontal mic coordinates are `(x, z)`
(the same axis swap the benchmark's Neural-SRP notebook applies). Near-field
delay-and-sum steering on the 84 STFT bins gives the **dominance score**
(brief Eq. 3)

    d_i = sum_tf | w(r_i, theta_i)^H X(t, f) |^2

which fixes the emission order: loudest first.

In [ ]:
# Mic XML -> (32, 3) [x_fwd, y_lat, z_up]: file stores (x, y_up, z_lat).
root = ET.parse(MIC_XML).getroot()
_pos = np.array([[float(p.get("x")), float(p.get("y")), float(p.get("z"))]
                 for p in root.iter("pos")])
assert _pos.shape == (N_MICS, 3), _pos.shape
MIC_POS = _pos[:, [0, 2, 1]]                   # -> (x_fwd, z_lat -> y, y_up -> z)
print("mic span x/y (m):", round(float(np.ptp(MIC_POS[:, 0])), 2),
      round(float(np.ptp(MIC_POS[:, 1])), 2))

# STFT bin frequencies: rfftfreq(512, 1/48000) masked to [100, 8000] Hz -> 84 bins.
_f_all = np.fft.rfftfreq(N_FFT, 1.0 / SR)
FREQS = _f_all[(_f_all >= FREQ_MIN_HZ) & (_f_all <= FREQ_MAX_HZ)]
assert len(FREQS) == FREQ_BINS, len(FREQS)

# Polar vocabulary: bin centers are the dequantized predictions.
R_CENTERS = (np.arange(N_R) + 0.5) * (MAX_RANGE_M / N_R)
TH_CENTERS = (np.arange(N_THETA) + 0.5) * (360.0 / N_THETA)   # [0, 360)

def ang_deg(x, y):
    return float(np.degrees(np.arctan2(y, x)) % 360.0)

def ang_diff(a, b):
    """Smallest absolute angular difference in degrees (benchmark _ad)."""
    return np.abs((np.asarray(a) - np.asarray(b) + 180.0) % 360.0 - 180.0)

def steering(r, theta_deg):
    """Unit-norm near-field steering vectors -> complex (P, F, M)."""
    r, t = np.atleast_1d(r).astype(float), np.radians(np.atleast_1d(theta_deg))
    src = np.stack([r * np.cos(t), r * np.sin(t), np.full_like(r, 1.0)], axis=1)
    d = np.maximum(np.linalg.norm(src[:, None] - MIC_POS[None], axis=2), 1e-3)
    a = np.exp(-2j * np.pi * FREQS[None, :, None] * d[:, None] / C_SOUND) / d[:, None]
    return a / np.linalg.norm(a, axis=2, keepdims=True)

def dominance(stft_complex, r, theta_deg):
    """Steered power at GT positions. stft_complex: (M=32, F=84, T)."""
    w = steering(r, theta_deg)
    y = np.einsum("pfm,mft->pft", np.conj(w), stft_complex)
    return np.sum(np.abs(y) ** 2, axis=(1, 2)).real

## Dataset

The pickle contract (same as the evidential dataset): `{"order": [tokens],
"items": {token: {stft (64, F=84, T), ann [{center: [x, y, z]}], velocity}}}`.
STFT channels are interleaved per mic — real `2i`, imag `2i+1` — and T is
padded/truncated to 19; the model consumes `(B, 64, T=19, F=84)`. Velocity
reduces to the linear speed scalar. Targets: `theta = atan2(y, x) % 360`,
`r = hypot(x, y)`, objects beyond 30 m dropped.

Token vocabulary `[PAD=0, BOS=1, EOS=2 | range x 24 | azimuth x 72]`; one
object = `(t_r, t_theta)`; emission order per `ORDERING` (dominance needs the
complex STFT, reconstructed from the interleaved channels). Everything is
precomputed once into tensors.

In [ ]:
PAD, BOS, EOS = 0, 1, 2
R_OFF, TH_OFF = 3, 3 + N_R
VOCAB = 3 + N_R + N_THETA
TPO = 2
MAX_LEN = 2 + MAX_OBJECTS * TPO

def encode_tokens(r, th, order):
    rb = np.clip((np.asarray(r) / (MAX_RANGE_M / N_R)).astype(int), 0, N_R - 1)
    tb = np.clip((np.asarray(th) / (360.0 / N_THETA)).astype(int), 0, N_THETA - 1)
    seq = [BOS]
    for i in order:
        seq += [R_OFF + int(rb[i]), TH_OFF + int(tb[i])]
    return np.array(seq + [EOS], dtype=np.int64)

def decode_tokens(toks):
    rs, ths, group = [], [], []
    for t in toks[1:]:
        if t in (EOS, PAD):
            break
        group.append(int(t))
        if len(group) == TPO:
            rb, tb = group[0] - R_OFF, group[1] - TH_OFF
            if 0 <= rb < N_R and 0 <= tb < N_THETA:
                rs.append(R_CENTERS[rb]); ths.append(TH_CENTERS[tb])
            group = []
    return np.array(rs), np.array(ths)

def speed_scalar(vel):
    """Velocity dict {'lin_x','lin_y'} or array -> linear speed (1,)."""
    if isinstance(vel, dict):
        return np.array([np.hypot(vel.get("lin_x", 0.0), vel.get("lin_y", 0.0))],
                        dtype=np.float32)
    v = np.asarray(vel, dtype=np.float32).ravel()
    return np.array([np.linalg.norm(v[:2])], dtype=np.float32)

def load_split(pkl_path, limit, rng):
    """Pickle -> dict of tensors: stft (N, 64, 19, 84), vel (N, 1),
    tokens (N, MAX_LEN), plus sample ids and per-scene GT for scoring."""
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)
    order = data["order"][:limit] if limit else data["order"]
    stfts, vels, toks, ids, gts = [], [], [], [], []
    t0 = time.time()
    for token in order:
        item = data["items"][token]
        stft = item.get("stft")
        if stft is None:
            continue
        stft = np.asarray(stft, dtype=np.float32)          # (64, F=84, T)
        T = stft.shape[2]
        stft = stft[:, :, :FIXED_T_FRAMES] if T >= FIXED_T_FRAMES else np.pad(
            stft, ((0, 0), (0, 0), (0, FIXED_T_FRAMES - T)))

        # targets: in-range sounding objects, theta in [0, 360)
        r_list, th_list = [], []
        for ann in (item.get("ann") or []):
            c = ann.get("center")
            if c is None:
                continue
            x, y = float(c[0]), float(c[1])
            r = float(np.hypot(x, y))
            if r > MAX_RANGE_M:
                continue
            r_list.append(r); th_list.append(ang_deg(x, y))
        r_arr, th_arr = np.array(r_list), np.array(th_list)

        # emission order (dominance needs the complex STFT: real 2i, imag 2i+1)
        if len(r_arr):
            if ORDERING == "dominance":
                X = stft[0::2] + 1j * stft[1::2]           # (32, 84, T=19)
                idx = np.argsort(-dominance(X, r_arr, th_arr))
            elif ORDERING == "near_to_far":
                idx = np.argsort(r_arr)
            elif ORDERING == "random":
                idx = rng.permutation(len(r_arr))
            else:
                raise ValueError(ORDERING)
        else:
            idx = np.zeros(0, int)

        seq = encode_tokens(r_arr, th_arr, idx)
        tok_row = np.full(MAX_LEN, PAD, np.int64)
        tok_row[:len(seq)] = seq[:MAX_LEN]

        stfts.append(stft.transpose(0, 2, 1))              # -> (64, T=19, F=84)
        vels.append(speed_scalar(item.get("velocity", {})))
        toks.append(tok_row)
        ids.append(token)
        gts.append({"r": r_arr, "theta": th_arr, "n": len(r_arr)})
    print(f"{Path(pkl_path).name}: {len(ids)} samples ({time.time() - t0:.0f}s)")
    return {"stft": torch.from_numpy(np.stack(stfts)), "vel": torch.from_numpy(np.stack(vels)),
            "tokens": torch.from_numpy(np.stack(toks)), "ids": ids, "gt": gts}

rng_data = np.random.default_rng(SEED)
data = {"train": load_split(TRAIN_PKL, LIMIT_TRAIN, rng_data),
        "val":   load_split(VAL_PKL, LIMIT_VAL, rng_data)}

def iter_batches(d, bs, shuffle, rng=None):
    idx = rng.permutation(len(d["ids"])) if shuffle else np.arange(len(d["ids"]))
    for k in range(0, len(idx), bs):
        sel = idx[k:k + bs]
        yield (d["stft"][sel].to(DEVICE_OBJ), d["vel"][sel].to(DEVICE_OBJ),
               d["tokens"][sel].to(DEVICE_OBJ), sel)

## Model

The MoG-slot front end, re-implemented faithfully (`blocks.py` / `network.py`
of `src/evidential`): SpeedFiLM2Ch_Factorized modulates the `(B, 64, 19, 84)`
input per (mic, re/im), per time frame, and per frequency bin from the ego
speed; the trunk runs `conv(1x7, fstride 2) -> conv(1x5, fstride 2) -> 2x
ResidualBlock(256) -> 1x1 -> n_theta=360 -> permute -> lazy 1x1 -> 100`
(unpadded, so F runs 84 -> 39 -> 18 before the permute), ending in the
`(B, 100, T=19, 360)` angular feature map.

The **only new part** is the head. The slot head pooled time and let K
learned queries cross-attend over the 360 angular tokens; SoundReg keeps the
identical token sequence — `mean over T -> (B, 360, 100)`, projected to
`d_model` + angular position embedding — and a causal Transformer decoder
cross-attends over it, emitting `(t_r, t_theta)` pairs until `[EOS]`.
Token-type masking keeps decoded sequences structurally valid.

In [ ]:
class ResidualBlock(nn.Module):
    """conv-bn-relu-conv-bn + skip, as in evidential blocks.py."""
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + x)

class SpeedFiLM2ChFactorized(nn.Module):
    """Velocity FiLM, factorized over (mic, re/im), time, and frequency.
    gamma/beta = sum of the three factors; x * (1 + gamma) + beta, so zero
    output = identity. Faithful to evidential blocks.py."""
    def __init__(self, time_steps, freq_bins, hidden=FILM_HIDDEN, mics=N_MICS):
        super().__init__()
        self.mics, self.T, self.F_ = mics, time_steps, freq_bins
        self.mlp = nn.Sequential(nn.Linear(1, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU())
        self.gamma_mic = nn.Linear(hidden, mics * 2)
        self.beta_mic = nn.Linear(hidden, mics * 2)
        self.gamma_t = nn.Linear(hidden, time_steps)
        self.beta_t = nn.Linear(hidden, time_steps)
        self.gamma_f = nn.Linear(hidden, freq_bins)
        self.beta_f = nn.Linear(hidden, freq_bins)
    def forward(self, x, speed):
        if speed is None:
            return x
        B = x.shape[0]
        h = self.mlp(speed.view(B, 1))
        xs = x.view(B, self.mics, 2, self.T, self.F_)
        gamma = (self.gamma_mic(h).view(B, self.mics, 2, 1, 1)
                 + self.gamma_t(h).view(B, 1, 1, self.T, 1)
                 + self.gamma_f(h).view(B, 1, 1, 1, self.F_))
        beta = (self.beta_mic(h).view(B, self.mics, 2, 1, 1)
                + self.beta_t(h).view(B, 1, 1, self.T, 1)
                + self.beta_f(h).view(B, 1, 1, 1, self.F_))
        return (xs * (1.0 + gamma) + beta).view_as(x)

class SoundReg(nn.Module):
    def __init__(self):
        super().__init__()
        # ---- reused MoG-slot front end ----
        self.film = SpeedFiLM2ChFactorized(FIXED_T_FRAMES, FREQ_BINS)
        self.conv1 = nn.Conv2d(64, 128, (1, 7), stride=(1, 2)); self.bn1 = nn.BatchNorm2d(128)
        self.conv2 = nn.Conv2d(128, 256, (1, 5), stride=(1, 2)); self.bn2 = nn.BatchNorm2d(256)
        self.res = nn.Sequential(ResidualBlock(256), ResidualBlock(256))
        self.conv3 = nn.Conv2d(256, N_THETA_FEAT, 1); self.bn3 = nn.BatchNorm2d(N_THETA_FEAT)
        # Lazy like the original: in_channels = F after the two unpadded
        # strided convs (84 -> 39 -> 18); let the first forward infer it.
        self.conv4 = nn.LazyConv2d(100, 1); self.bn4 = nn.BatchNorm2d(100)

        # ---- new head: causal decoder over the angular tokens ----
        self.mem_proj = nn.Linear(100, D_MODEL)
        self.mem_pos = nn.Embedding(N_THETA_FEAT, D_MODEL)
        self.tok_emb = nn.Embedding(VOCAB, D_MODEL)
        self.pos_emb = nn.Embedding(MAX_LEN, D_MODEL)
        layer = nn.TransformerDecoderLayer(D_MODEL, N_HEADS, D_FF, DROPOUT,
                                           batch_first=True, norm_first=True)
        # norm_first leaves the stream unnormalized; without the final norm
        # the tied logits start at CE ~ 100 instead of ln(VOCAB) ~ 4.6.
        self.dec = nn.TransformerDecoder(layer, N_LAYERS, norm=nn.LayerNorm(D_MODEL))
        self.head = nn.Linear(D_MODEL, VOCAB, bias=False)
        self.head.weight = self.tok_emb.weight
        for emb in (self.tok_emb, self.pos_emb, self.mem_pos):
            nn.init.normal_(emb.weight, std=0.02)

    def encode(self, stft, speed):
        """(B, 64, 19, 84) + (B, 1) -> memory (B, 360, D_MODEL).
        Identical flow to MoGSlotNetwork up to the (B, 100, T, 360) map,
        then the AttentionSlotHead's own tokenization: mean over T."""
        x = self.film(stft, speed)
        x = F.relu(self.bn1(self.conv1(x)))            # (B, 128, 19, 39)
        x = F.relu(self.bn2(self.conv2(x)))            # (B, 256, 19, 18)
        x = self.res(x)
        x = F.relu(self.bn3(self.conv3(x)))            # (B, 360, 19, 18)
        x = x.permute(0, 3, 2, 1)                      # (B, 18, 19, 360)
        x = F.relu(self.bn4(self.conv4(x)))            # (B, 100, 19, 360)
        tokens = x.mean(dim=2).permute(0, 2, 1)        # (B, 360, 100)
        return self.mem_proj(tokens) + self.mem_pos.weight[None]

    def forward(self, stft, speed, tokens):
        """Teacher forcing: logits for tokens[:, 1:] given tokens[:, :-1]."""
        mem, inp = self.encode(stft, speed), tokens[:, :-1]
        L = inp.shape[1]
        causal = torch.triu(torch.ones(L, L, dtype=torch.bool, device=inp.device), 1)
        x = self.tok_emb(inp) + self.pos_emb.weight[None, :L]
        out = self.dec(x, mem, tgt_mask=causal, tgt_key_padding_mask=(inp == PAD))
        return self.head(out)

def valid_mask(step, device):
    """Legal next tokens at generation step (0 = first after BOS): range
    steps emit range tokens, azimuth steps azimuth tokens, EOS only at
    object boundaries."""
    m = torch.zeros(VOCAB, dtype=torch.bool, device=device)
    if step % TPO == 0:
        m[EOS] = True
        m[R_OFF:R_OFF + N_R] = True
    else:
        m[TH_OFF:TH_OFF + N_THETA] = True
    return m

@torch.no_grad()
def greedy_decode(model, stft, speed):
    """Type-masked greedy decoding. Returns per scene: (r, theta) arrays,
    per-object confidence exp(mean token logprob), and P(EOS) at every
    object boundary (the EOS-calibration record)."""
    model.eval()
    B = stft.shape[0]
    mem = model.encode(stft, speed)
    toks = torch.full((B, 1), BOS, dtype=torch.long, device=stft.device)
    done = torch.zeros(B, dtype=torch.bool, device=stft.device)
    lps = [[] for _ in range(B)]
    eos_probs = [[] for _ in range(B)]
    for step in range(MAX_OBJECTS * TPO + 1):
        L = toks.shape[1]
        causal = torch.triu(torch.ones(L, L, dtype=torch.bool, device=stft.device), 1)
        x = model.tok_emb(toks) + model.pos_emb.weight[None, :L]
        out = model.dec(x, mem, tgt_mask=causal, tgt_key_padding_mask=(toks == PAD))
        logits = model.head(out[:, -1]).masked_fill(~valid_mask(step, stft.device)[None], -1e9)
        logp = logits.log_softmax(-1)
        if step % TPO == 0:
            pe = logp[:, EOS].exp()
            for i in range(B):
                if not done[i]:
                    eos_probs[i].append(float(pe[i]))
        if step < MAX_OBJECTS * TPO:
            nxt = logits.argmax(-1)
        else:  # capacity reached: force EOS on whoever is still going
            nxt = torch.full((B,), EOS, dtype=torch.long, device=stft.device)
        step_lp = logp.gather(1, nxt[:, None]).squeeze(1)
        for i in range(B):
            if not done[i]:
                lps[i].append(float(step_lp[i]))
        nxt = torch.where(done, torch.full_like(nxt, PAD), nxt)
        toks = torch.cat([toks, nxt[:, None]], 1)
        done |= nxt == EOS
        if bool(done.all()):
            break
    out = []
    for i in range(B):
        r, th = decode_tokens(toks[i].cpu().numpy())
        conf = [float(np.exp(np.mean(lps[i][k * TPO:(k + 1) * TPO])))
                for k in range(len(r))]
        out.append({"r": r, "theta": th, "conf": np.array(conf),
                    "eos_probs": np.array(eos_probs[i])})
    return out

model = SoundReg().to(DEVICE_OBJ)
# One dummy forward materializes the LazyConv2d before the optimizer is built.
with torch.no_grad():
    model.encode(torch.zeros(1, 64, FIXED_T_FRAMES, FREQ_BINS, device=DEVICE_OBJ),
                 torch.zeros(1, 1, device=DEVICE_OBJ))
print("params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## Training

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
steps_total = EPOCHS * math.ceil(len(data["train"]["ids"]) / BATCH_SIZE)
warm = max(1, int(0.05 * steps_total))
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: (s + 1) / warm if s < warm
    else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps_total - warm))))
rng_epoch = np.random.default_rng(SEED)

def run_epoch(train):
    model.train(train)
    tot, nt = {"loss": 0.0, "acc": 0.0}, 0
    d = data["train"] if train else data["val"]
    for stft, vel, toks, _ in iter_batches(d, BATCH_SIZE, train, rng_epoch):
        logits, tgt = model(stft, vel, toks), toks[:, 1:]
        loss = F.cross_entropy(logits.reshape(-1, VOCAB), tgt.reshape(-1), ignore_index=PAD)
        if train:
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step(); sched.step()
        m = tgt != PAD
        bs = stft.shape[0]
        tot["loss"] += float(loss) * bs
        tot["acc"] += float((logits.argmax(-1)[m] == tgt[m]).float().mean()) * bs
        nt += bs
    return {k: v / nt for k, v in tot.items()}

best_val, history = float("inf"), []
for epoch in range(1, EPOCHS + 1):
    tr = run_epoch(True)
    with torch.no_grad():
        va = run_epoch(False)
    row = {"epoch": epoch, **{f"train_{k}": v for k, v in tr.items()},
           **{f"val_{k}": v for k, v in va.items()}}
    history.append(row); print(row)
    torch.save({"model": model.state_dict(), "epoch": epoch}, RUN_DIR / "last.pt")
    if va["loss"] < best_val:
        best_val = va["loss"]
        torch.save({"model": model.state_dict(), "epoch": epoch}, RUN_DIR / "best.pt")
with open(RUN_DIR / "history.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(history[0].keys())); w.writeheader(); w.writerows(history)
print("best checkpoint:", RUN_DIR / "best.pt")

## Training curves

In [ ]:
ep = [r["epoch"] for r in history]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(ep, [r["train_loss"] for r in history], label="train")
ax[0].plot(ep, [r["val_loss"] for r in history], label="val")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("token CE"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ep, [r["train_acc"] for r in history], label="train")
ax[1].plot(ep, [r["val_acc"] for r in history], label="val")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("next-token acc"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Inference and prediction export

Best checkpoint, greedy decoding on the four benchmark test splits.
Predictions go to `pred_soundreg_<split>.csv` in the benchmark format —
`confidence` is the per-object mean token probability (diagnostic only;
cardinality came from `[EOS]`, not from a threshold).

In [ ]:
model.load_state_dict(torch.load(RUN_DIR / "best.pt", map_location=DEVICE_OBJ,
                                 weights_only=True)["model"])
PRED_FIELDS = ["sample_id", "model_name", "confidence", "theta_deg", "range_m",
               "class_name", "sigma_theta_deg", "sigma_r_m"]

test_data, eos_records = {}, []
for sp, pkl in SPLITS.items():
    d = load_split(pkl, LIMIT_TEST, np.random.default_rng(SEED))
    test_data[sp] = d
    rows = []
    for stft, vel, _, sel in iter_batches(d, BATCH_SIZE, False):
        for j, pred in enumerate(greedy_decode(model, stft, vel)):
            token = d["ids"][sel[j]]
            n_gt = d["gt"][sel[j]]["n"]
            eos_records += [(p, float(k >= n_gt)) for k, p in enumerate(pred["eos_probs"])]
            for r, th, cf in zip(pred["r"], pred["theta"], pred["conf"]):
                rows.append({"sample_id": token, "model_name": MODEL_NAME,
                             "confidence": f"{cf:.6f}", "theta_deg": f"{th % 360.0:.6f}",
                             "range_m": f"{r:.6f}", "class_name": "object",
                             "sigma_theta_deg": "", "sigma_r_m": ""})
    with open(RESULTS_DIR / f"pred_{MODEL_NAME}_{sp}.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=PRED_FIELDS); w.writeheader(); w.writerows(rows)
    print(f"[{sp}] {len(d['ids'])} samples -> {len(rows)} predictions")

## Evaluation

The benchmark's greedy one-to-one matcher, verbatim: angle gate 5 deg, and
the joint 5 deg + 5 m gate for range-capable models. Metrics per split:
P / R / F1, mAAE (matched angle error), mADE (matched range error) — the same
columns as `benchmark_table_full.csv`, so SoundReg rows drop straight into
the comparison. The EOS Brier score (P(EOS) vs should-have-stopped) is the
early-stopping diagnostic: high values with low recall on `*_together`
splits = the decoder is dropping quiet sources.

In [ ]:
def score(gts_by_id, preds_by_id, use_range):
    """Benchmark scoring.py score_at_gate, inlined."""
    tp = fp = fn = 0
    ae, re_ = [], []
    for t in set(gts_by_id) | set(preds_by_id):
        g = list(gts_by_id.get(t, []))
        used = set()
        for (pa, prng) in preds_by_id.get(t, []):
            best, bj = None, None
            for j, (ga, gr) in enumerate(g):
                if j in used:
                    continue
                ea = float(ang_diff(pa, ga))
                if ea > ANGLE_GATE:
                    continue
                if use_range:
                    er = abs(prng - gr)
                    if er > RANGE_GATE:
                        continue
                    cost = ea + er
                else:
                    cost = ea
                if best is None or cost < best:
                    best, bj = cost, j
            if bj is not None:
                used.add(bj); tp += 1
                ae.append(float(ang_diff(pa, g[bj][0])))
                if use_range:
                    re_.append(abs(prng - g[bj][1]))
            else:
                fp += 1
        fn += len(g) - len(used)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    out = {"tp": tp, "fp": fp, "fn": fn, "precision": round(prec, 4),
           "recall": round(rec, 4), "f1": round(f1, 4),
           "mAAE_deg": round(float(np.mean(ae)), 4) if ae else float("nan")}
    if use_range:
        out["mADE_m"] = round(float(np.mean(re_)), 4) if re_ else float("nan")
    return out

all_rows = []
for sp in SPLITS:
    d = test_data[sp]
    gts_by_id = {t: list(zip(g["theta"], g["r"])) for t, g in zip(d["ids"], d["gt"])}
    preds_by_id = {}
    for r in csv.DictReader(open(RESULTS_DIR / f"pred_{MODEL_NAME}_{sp}.csv")):
        preds_by_id.setdefault(r["sample_id"], []).append(
            (float(r["theta_deg"]), float(r["range_m"])))
    all_rows.append({"model": MODEL_NAME, "split": sp, "gate": "5deg",
                     **score(gts_by_id, preds_by_id, False)})
    all_rows.append({"model": MODEL_NAME, "split": sp, "gate": "5deg+5m",
                     **score(gts_by_id, preds_by_id, True)})

keys = []
for m in all_rows:
    for k in m:
        if k not in keys:
            keys.append(k)
with open(METRICS_CSV, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=keys); w.writeheader()
    for m in all_rows:
        w.writerow({k: m.get(k, "") for k in keys})
for m in all_rows:
    print(m)

p_eos = np.array([p for p, _ in eos_records]); y = np.array([s for _, s in eos_records])
print(f"\nEOS Brier {np.mean((p_eos - y) ** 2):.4f} | "
      f"mean P(EOS) when should continue {p_eos[y == 0].mean():.3f}, "
      f"when should stop {p_eos[y == 1].mean():.3f}")
print("wrote", METRICS_CSV)

## Result

Outputs: `notebooks/runs/soundreg_v1/{history.csv, best.pt, last.pt}` and
benchmark-format CSVs in `notebooks/results/` (`pred_soundreg_<split>.csv`,
`metrics_soundreg.csv`) — rows directly comparable to
`benchmark_table_full.csv` (evidential MoG-slot: F1 0.5515 @5deg on
static_easy, 0.4706 @5deg+5m).

**Read the `*_together` splits first** — multi-source scenes are where the
dominance-order conditioning is supposed to pay. If recall there is weak,
check the EOS line above before blaming the idea (early-EOS is the known
failure mode).

**Next knobs (one Settings change each):** `ORDERING = "random"` /
`"near_to_far"` (does dominance specifically matter?); freeze-vs-train the
reused trunk; `N_THETA`/`N_R` vocabulary resolution; beam / sampled decoding.